### Restart and Run All Cells

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
year = 2025
quarter = 4
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:12 13:27:47


In [4]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',   
                'latest_amt':'{:,}','previous_amt':'{:,}','inc_amt':'{:,}',
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%','inc_pct':'{:.2f}%'
              }

In [8]:
sql = f"""
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = {year} AND quarter <= {quarter}) 
ORDER BY year DESC, quarter DESC
"""
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2025 AND quarter <= 4) 
ORDER BY year DESC, quarter DESC



In [10]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.shape

(201, 5)

In [14]:
sql = f"""
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = {year} AND quarter <= {quarter}-1) 
OR (year = {year}-1 AND quarter >= {quarter}) 
ORDER BY year DESC, quarter DESC"""
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2025 AND quarter <= 4-1) 
OR (year = 2025-1 AND quarter >= 4) 
ORDER BY year DESC, quarter DESC


In [16]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8099,10,"7,719,299",4
1,ACE,8099,10,"839,069",4
2,ADVANC,8099,10,"42,863,183",4
3,AEONTS,8099,10,"2,907,113",4
4,AH,8099,10,"745,761",4


In [18]:
dfp.name.unique().shape

(221,)

In [20]:
sql = """
SELECT *
FROM stocks
"""
stocks = pd.read_sql(sql, conlt)
stocks.shape

(226, 15)

In [22]:
sql = """
SELECT *
FROM tickers
"""
tickers = pd.read_sql(sql, conlt)
tickers.shape

(394, 9)

In [24]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ACE,2025,Q4,"798,614","839,069","-40,455",-4.82%
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
3,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%
4,AIE,2025,Q4,"21,904","168,287","-146,383",-86.98%


In [26]:
# Create the SQL query with parameter binding
sqlDel = f"DELETE FROM qt_profits WHERE year = {year} AND quarter = 'Q{quarter}'"
print(sqlDel)
rp = conlt.execute(text(sqlDel))
# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

DELETE FROM qt_profits WHERE year = 2025 AND quarter = 'Q4'
Rows deleted: 141


In [28]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
tickers.sample(5)

,name,id
276,SHANG,442
147,INSET,738
116,FSS,179
372,UAC,592
336,TIF1,718


In [30]:
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
df_ins.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%,234
1,ACE,2025,Q4,"798,614","839,069","-40,455",-4.82%,698
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%,6
3,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%,9
4,AIE,2025,Q4,"21,904","168,287","-146,383",-86.98%,720


In [32]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO qt_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [34]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ACE,2025,Q4,"798,614","839,069","-40,455",-4.82%
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
3,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%
5,AIMIRT,2025,Q4,"684,130","830,452","-146,322",-17.62%


In [36]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
1,ACE,2025,Q4,"798,614","839,069","-40,455",-4.82%
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
3,AH,2025,Q4,"731,428","745,761","-14,333",-1.92%
5,AIMIRT,2025,Q4,"684,130","830,452","-146,322",-17.62%


In [38]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
7,AJ,2025,Q4,"-553,329","-855,675","302,346",35.33%
13,ASK,2025,Q4,"531,545","388,249","143,296",36.91%
15,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%


In [40]:
df_ins_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[df_ins_criteria, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%
15,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%
23,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%
36,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%


In [42]:
df_ins[df_ins_criteria].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
53,DIF,2025,Q4,"13,855,643","868,268","12,987,375",1495.78%,140
23,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%,52
155,SPRC,2025,Q4,"2,569,354","1,641,889","927,465",56.49%,470
36,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%,74
48,CPNREIT,2025,Q4,"3,460,976","2,333,575","1,127,401",48.31%,647


In [44]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%,234
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%,6
15,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%,728
23,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%,52
36,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%,74


In [46]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2025,Q4,"10,827,771","7,719,299","3,108,472",40.27%,234
2,ADVANC,2025,Q4,"47,885,902","42,863,183","5,022,719",11.72%,6
15,ASW,2025,Q4,"1,077,662","846,075","231,587",27.37%,728
23,BCP,2025,Q4,"2,879,701","679,670","2,200,031",323.69%,52
36,BPP,2025,Q4,"3,026,277","1,960,505","1,065,772",54.36%,74


In [48]:
conlt.commit()
conlt.close()

In [50]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:12 13:37:58
